### Import statements

In [ ]:
# Basic imports to set up circuits (more may be needed for specific additional functionality)
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit import Parameter # To enable parameterized circuits
from qiskit.quantum_info import SparsePauliOp # Used for example to set up observables for Estimator

# Basic import to enable transiplation
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Basic imports to set up backends and run circuits either via Estimator or Sample
from qiskit_ibm_runtime import EstimatorV2 as Estimator 
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService

# Imports to allow setting job_tags when submitting jobs
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.options import SamplerOptions
from qiskit_ibm_runtime.options import EnvironmentOptions

# Imports for math and data analysis
import numpy as np
from numpy import pi
from matplotlib import pyplot as plt

### Backend setup

In [ ]:
# Set up a backend that runs a noiseless simulator
from qiskit_aer import AerSimulator
backend = AerSimulator()

# Set up a fake_provider backend to run locally (e.g., emulating the ibm_marrakesh QPU)
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
backend = FakeMarrakesh()

# Set up a real QPU backend on IBM Quantum Platform (see https://quantum.cloud.ibm.com/docs/en/guides/initialize-account for more info)
# You'll need to create the API key by clicking "Create +" on the left side of the IBM Quantum dashboard main page, and copy the text here
# You can copy the CRN to the clipboard by clicking the double rectangle icon next to Qiskit-Open under Instances (also on the left side of the dashboard home page)
service = QiskitRuntimeService(channel="ibm_quantum_platform", token="<your API Key>", instance="<your CRN>")
print(service.backends())

# Select least busy QPU
backend = service.least_busy(simulator=False, operational=True)

# Alternatively, choose particular backend (e.g., ibm_torino)
backend = service.backend('ibm_torino')

# Display the selected backend for confirmation (always good to check that you're not accidentally running on a $$$ QPU)
print(backend)

### Transpiler setup

In [ ]:
# Create the "pass manager" that will perform the transpilation
pm = generate_preset_pass_manager(backend=backend, optimization_level=0)

### Estimator or Sampler setup

In [ ]:
# Store the options you wish to specify for an Estimator
my_estimator_options = EstimatorOptions()
my_estimator_options.environment = EnvironmentOptions(job_tags=['<ENTER YOUR JOB TAGS (SUCH AS PROBLEM NAME) HERE'])

# Set up estimator; skip options if running on simulator
estimator = Estimator(backend, options=my_estimator_options)

# Store the options you wish to specify for a Sampler
my_sampler_options = SamplerOptions()
my_sampler_options.environment = EnvironmentOptions(job_tags=['<ENTER YOUR JOB TAGS (SUCH AS PROBLEM NAME) HERE'])

# Set up sampler; skip options if running on simulator
sampler = Sampler(backend, options=my_sampler_options)


### Useful commands for circuit creation

In [ ]:
# Set number of data qubits
num_data_qubits = 2

# Create qubit and classical data bit registers (you can create and name additional registers if it's useful for bookkeeping)
data_qubits = QuantumRegister(num_data_qubits, "qubit")
classical_data = ClassicalRegister(num_data_qubits, "classical")

# Create circuit object
qc = QuantumCircuit(data_qubits,  classical_data)

# Add a (Hadamard in this example) gate on a specific qubit
qc.h(data_qubits[0])

# Add a barrier to prevent transipler from collapsing gates between different parts of the circuit
# To place a barrier on specific qubits, a list of qubits can be given as the argument inside the ( )'s
qc.barrier() # Place barrier across entire circuit

# Create parametrized gate
phi = Parameter('$\\phi$')
qc.rz(phi, data_qubits[1]) # Perform a z rotation through angle phi on qubit qdata[1]

# Introduce a time delay of duration tau.
# You must select the units in which you are specifying the delay duration.
# While Qiskit allows you to specify the delay directly in ns or us (microseconds), this causes problems when
# running on real QPUs that have a fixed clock rate (particularly if your delay doesn't line up with the clock cycle)
# *** I strongly recommend to use unit = 'dt' to specify tau directly in terms of the number of clock cycles. ***
# Use the first line below to find the value of dt for your chosen backend, which will allow you to convert between real time and clock cycles
print(backend.dt) # Print the value of dt for the chosen backend.
tau = Parameter('$\\tau$')
qc.delay(tau, unit = 'dt')

# Assign specific value to parameter and store in new circuit
# "inplace = False" prevents the assign_parameters method from overwriting circuit qc with specific parameter values
# "inplace = True" can be used to simply overwrite qc with the parameter values substituted in. 
# Arrays of parameter values can also/instead be supplied to Estimator.run and Sampler.run directly (see below).
assigned_circuit = qc.assign_parameters({phi: pi/12}, inplace=False)

# Measure a specific qubit or list of qubits and send result to a specific classical data bit or list of classical data bits
qc.measure(data_qubits[0], classical_data[0])

# Measure all qubits (e.g., at the end of a circuit when using the Sampler Primitive)
# This method adds a new classical register called "meas" to store all of the measurement results
qc.measure_all()

# If you already have a classical register with at least as many bits as there are qubits in the circuit, you can use
# the following to send the measurement results directly to your own classical register.
qc.measure_all(add_bits=False)

# Create conditional logic with "if_test" statement to determine next gates based on data in classical register
# The following example applies a Y gate to data_qubit 1 if the measurement outcome stored in the classical register is "00" 
# Note that the "0b" prefix tells the notebook to interpret the numbers that follow as a bit string
with qc.if_test((classical_data, 0b00)):
    qc.y(data_qubits[1])

# The if_test statement can also be set up with an "else" clause, but note that nested if statements are NOT supported.
# The following example applies a Hadamard gate to data qubit 0 if the classical register holds "11," otherwise an X gate is applied
with qc.if_test((classical_data, 0b11)) as else_:
    qc.h(data_qubits[0])
with else_:
    qc.x(data_qubits[0])


### Transpile circuit

In [ ]:
# Run the pass manager to complete the transpilation
transpiled_circuit = pm.run(qc)

# Display the transpiled circuit
# The idle_wires = False argument will suppress all of the unused qubits from showing
transpiled_circuit.draw('mpl', idle_wires = False)

### Run Estimator or Sampler

In [ ]:
# For Estimator, first set up observables to be measured
observables_labels = ["IZ", "ZI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

# Map observables onto transpiled circuit layout
mapped_observables = [observable.apply_layout(transpiled_circuit.layout) for observable in observables]

# Run Estimator for non-parametrized circuit
job = estimator.run([(transpiled_circuit, mapped_observables)])

# Run Sampler 
job = sampler.run([transpiled_circuit])

# Run Sampler with specified number of shots (e.g., 2048)
job = sampler.run([transpiled_circuit], shots = 2048)

# Run Sampler on parameterized circuit over list of specified values
parameter_values = np.array([0, 1, 2])
job = sampler.run([(transpiled_circuit, parameter_values)]) 

# Optionally, you can specify the number of shots to use for each fixed parameter values (this also works for non-parameterized circuits)
parameter_values = np.array([0, 1, 2])
N_shots = 2048
job = sampler.run([(transpiled_circuit, parameter_values, N_shots)]) 

### Special considerations for using Estimator with parameterized circuits
If you want to run Estimator on a parameterized circuit, there are some subtleties about how the parameter value and observable arrays must be organized. The following links have useful information on this: 

https://docs.quantum.ibm.com/guides/primitive-input-output

https://docs.quantum.ibm.com/guides/primitives-examples. 

In particular, see this section on the "broadcasting rules:" 

https://quantum.cloud.ibm.com/docs/en/guides/primitive-input-output#broadcasting-rules.

In the following example code, the observables are given as a *column* vector and the parameters as a *row* vector. According to the broadcasting rules, this will give output which is a two-dimensional array containing the results for all observables for each of the specified parameter values. This is the situation I expect will be most useful for general use. In the output, each column will be the set of expectation values for a fixed parameter value. 

In [ ]:
# Convert list of observables into a 4x1 column vector (here 4 refers to the 4 observables specified above) 
reshaped_observables = np.fromiter(mapped_observables, dtype=object)
reshaped_observables = reshaped_observables.reshape((4, 1))

# Specify list of parameter values 
parameter_values = np.array([0, 1, 2])

# Reshape into 1x3 row vector
parameter_values = parameter_values[np.newaxis,:]

# Run the Estimator over all of the specified parameter values
job = estimator.run([(transpiled_circuit, reshaped_observables, parameter_values)])

### Retrieval of results
Use these command at the end of your code to retrieve and analyze results.
Here it is assumed that the Estimator or Sampler was run via

`job = estimator.run(...)` or `job = sampler.run(...)` 

In [ ]:
# Store results
job_result = job.result()
 
# This is the result from a single pub, and holds results for all observables estimated (if using Estimator) 
pub_result = job.result()[0]

# For Estimator: extract data values and standard deviations
values = pub_result.data.evs
errors = pub_result.data.stds

# For Sampler: extract measurement counts 
# "classical" below refers to the name given above to the classical register. If no name is given, the default is "meas:" data.meas.get_counts()    
measurement_counts = pub_result.data.classical.get_counts()

# To extract the number of counts for a particular bit string (e.g., 01101) use
measurement_counts.get("01101")

# If Sampler is run on a parametrized circuit, the commands above will give counts aggregated over all runs with all parameter values looped over
# To extract a list of separate results for each of the parameter values used:
measurement_counts = [pub_result.data.classical.get_counts(param_index) for param_index in range( 0, len(parameter_values) ) ]

# To extract number of counts for a particular bit string (e.g., 01101) and for a specific parameter value (e.g., corresponding to index 2 in the list parameter_values), use
measurement_counts[2].get("01101") # Similar to above, you can also use Python list making operations to extract the count information directly into lists/arrays

